# LLM Learned Operations

Small LLMs are notoriously bad with maths (which is funny because one forward pass through a transfomer model has many many matrix multiplications and additions!). We can use our `SymbolicModel` to probe what functions a small LLM is actually using when carrying out mathematical operations.

In this demo, we use the small model Llama-3.2-1B-Instruct. Depending on your laptop, you should be able to run this whole notebook locally!

## Set-up

In [1]:
import numpy as np
from symtorch import SymbolicModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# This is the model we are going to use
model_name = "meta-llama/Llama-3.2-1B-Instruct"

# Load the tokenizer and the model
tok = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.generation_config.pad_token_id = tok.eos_token_id

torch.manual_seed(290402)
# For our experiment, we want a deterministic model
torch.use_deterministic_algorithms(True)

# Function which calls our LLM
def llm_call(prompt: str, max_tokens = 250) -> str:
    inputs = tok(prompt, return_tensors="pt")

    out = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        do_sample=False,          # greedy
    )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

Let's try out our LLM to see how it performs at basic addition.

In [3]:
output = llm_call("Return only the numeric answer in the format $boxed$. What is 12+7=?")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [4]:
print(output)

.

## Step 1: We need to add 12 and 7 together.
## Step 2: The result of the addition is 19.
## Step 3: We need to put the result in the format $boxed$.
## Step 4: The final answer is $\boxed{19}$.

The final answer is: $\boxed{19}$


For smaller numbers it can perform reasonably well. Let's see it's behvaiour for larger (3 digit) numbers.

In [5]:
output = llm_call("Return only the numeric answer in the format $boxed$. What is 972+373=?")
print(output)

print("True answer = ", 972+373)

## Step 1: Add the two numbers together
First, we need to add 972 and 373 together.

## Step 2: Calculate the sum
972 + 373 = 1445

## Step 3: Format the answer
The answer should be in the format $boxed{1445}$.

The final answer is: $\boxed{1445}$
True answer =  1345


No longer performs that great! 

We can use `SymbolicModel` to approximate the functions that the LLM is using when performing maths.

In [6]:
# Get out the number outputted by llm as float
def extract_boxed_number(text: str) -> float:
    def parse_number(s: str) -> float:
        return float(s.replace(',', ''))
    
    # Try $\boxed{...}$ format first
    match = re.search(r'\$\\boxed\{([^}]+)\}\$', text)
    if match:
        return parse_number(match.group(1))
    # Try $boxed{...}$ format (without backslash)
    match = re.search(r'\$boxed\{([^}]+)\}\$', text)
    if match:
        return parse_number(match.group(1))
    # Try \boxed{...} without dollar signs
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    if match:
        return parse_number(match.group(1))
    # Try boxed{...} without anything
    match = re.search(r'boxed\{([^}]+)\}', text)
    if match:
        return parse_number(match.group(1))
    # Try $number$ format (without boxed)
    match = re.search(r'\$([0-9,.]+)\$', text)
    if match:
        return parse_number(match.group(1))
    # Fallback: try to find any number after an equals sign
    match = re.search(r'=\s*([\d,]+)', text)
    if match:
        return parse_number(match.group(1))
    # Fallback: try to find "Answer: number"
    match = re.search(r'Answer:\s*([\d,.]+)', text)
    if match:
        return parse_number(match.group(1))
    raise ValueError(f"No boxed number found in: {text}")

# Function to create a dataset of random number pairs 
def random_number_pairs(N = 100, maximum = 999):
    return np.random.randint(0, maximum, size=(N, 2))

## Addition

`SymbolicModel` is model-agnostic. You just need to pass a function that is of the form `f(inputs) = outputs`.

In [7]:
# Create a function that the SymbolicModel expects 
def llm_addition(X):
    outputs = []
    # X is of shape (N,2)
    for n in range(X.shape[0]):
        a = X[n,0]
        b = X[n,1]
        output = llm_call(f"Return only the numeric answer in the format $boxed$. What is {int(a)}+{int(b)}=?")
        output = extract_boxed_number(output)
        outputs.append(output)
    return np.array(outputs)

Create a random dataset of numbers to add.

In [8]:
np.random.seed(290402)

X = random_number_pairs(50)

Example of the numbers in our dataset.

In [9]:
print(X[:5,:])

[[451  41]
 [871 582]
 [237 193]
 [661 992]
 [417 724]]


In [10]:
# Initialise our model
symbolic_model_addition = SymbolicModel(llm_addition, block_name = "llm_addition_func")

In [11]:
sr_params = {'constraints': {'sin':1, 'exp':1}, 'niterations' : 1000}

In [12]:
#Perform SR on our model
symbolic_model_addition.distill(X, sr_params= sr_params)

/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
Compiling Julia backend...


🛠️ Running SR on output dimension 0 of 0


[ Info: Started!



Expressions evaluated per second: 2.380e+06
Progress: 10707 / 31000 total iterations (34.539%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.522e+05  0.000e+00  y = 1002.9
3           8.858e+03  1.414e+00  y = x₁ + x₀
5           8.173e+03  4.001e-02  y = (x₁ * 0.95782) + x₀
7           7.936e+03  1.442e-02  y = ((x₁ * 0.89627) + x₀) * 1.0399
9           7.932e+03  1.577e-05  y = (((x₁ * 0.89719) + -5.5603) + x₀) * 1.0447
10          1.660e+03  1.564e+00  y = (x₀ + inv((x₀ * -0.005024) + 0.34498)) + x₁
12          1.510e+03  4.739e-02  y = x₀ + ((x₁ * 0.98066) + inv((x₀ * -0.003442) + 0.23577)...
                                      )
14          1.465e+03  1.486e-02  y = ((x₁ * 0.97429) + (inv((x₀ * -0.0031489) + 0.21556) + ...
                                      7.7701)) 

[ Info: Final population:
[ Info: Results saved to:


{0: PySRRegressor.equations_ = [
 	    pick     score                                           equation  \
 	0         0.000000                                          1002.8534   
 	1         1.422072                                            x1 + x0   
 	2         0.040272                             x0 + (x1 * 0.95783037)   
 	3         0.014673                (x1 + (x0 * 1.1157405)) * 0.9320593   
 	4         0.000260  (((x1 * 0.89719146) + -5.5602617) + x0) * 1.04...   
 	5         1.566168   x0 + (x1 + inv((x0 * -0.012401185) + 0.8540098))   
 	6         0.051629  (inv((x0 * -0.0071331137) + 0.49047625) + x0) ...   
 	7         0.019960  ((x0 + (x1 * 0.9611138)) + 9.976891) + inv((x0...   
 	8         0.017441  (x0 + (inv((x0 * -0.007133195) + 0.49047813) +...   
 	9         0.031700  ((x1 + inv((x0 * -0.007133231) + 0.49048457)) ...   
 	10        0.082518  ((x0 * -0.007133387) * inv(sin(x0))) + (x0 + (...   
 	11  >>>>  0.092871  ((((inv(sin(x0)) * (x0 * -8.280994e-6)) + 0.9

  - SR_output/llm_addition_func/dim0_1764345478/hall_of_fame.csv


In [15]:
symbolic_model_addition.show_symbolic_expression(complexity=[3])


➡️ Dimension 0 - Complexity 3:
   x1 + x0 (loss: 8.858000e+03)


In [17]:
symbolic_model_addition.show_symbolic_expression()


➡️ Standard symbolic expressions for output dimension 0:
    complexity          loss  \
0            1  152241.12000   
1            3    8858.00000   
2            5    8172.51950   
3            7    7936.18260   
4            8    5539.63570   
5           10    1634.10170   
6           11    1587.55290   
7           12    1507.41760   
8           13    1477.81480   
9           14    1379.15530   
10          16    1229.54610   
11          18    1178.01400   
12          20    1151.30570   
13          21    1131.40780   
14          22    1008.77200   
15          24     863.80830   
16          26     795.50726   
17          27     783.65510   
18          28     712.91790   
19          30     704.59010   

                                             equation     score  \
0                                           1002.8534  0.000000   
1                                             x1 + x0  1.422072   
2                              (x1 * 0.95781434) + x0  0.040272   
3

`symbolic_model_addition` contains a list of equations. The more complex equations fit the inputs $\rightarrow$ outputs better, but may overfit. The 'best equation' is the one that balances complexity and accuracy the most (largest gain in accuracy per increase in complexity).

Let's see how the LLM performs other tasks.

## Multiplication

In [18]:
def llm_multiplication(X):
    outputs = []
    # X is of shape (N,2)
    for n in range(X.shape[0]):
        a = X[n,0]
        b = X[n,1]
        output = llm_call(f"Return only the numeric answer in the format $boxed$. What is {int(a)} * {int(b)}=?")
        output = extract_boxed_number(output)
        outputs.append(output)
    return np.array(outputs)

In [19]:
# Initialise our model
symbolic_model_multiplication = SymbolicModel(llm_multiplication)

No name specified for this block. Label is block_17429733968.


In [20]:
#Perform SR on our model
symbolic_model_multiplication.distill(X, sr_params=sr_params)

🛠️ Running SR on output dimension 0 of 0


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 2.590e+06
Progress: 10862 / 31000 total iterations (35.039%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           4.527e+10  0.000e+00  y = 2.2987e+05
3           1.676e+10  -0.000e+00  y = x₁ * x₀
5           1.429e+10  -0.000e+00  y = (x₁ * 0.85344) * x₀
7           1.361e+10  -0.000e+00  y = ((x₀ * 0.77067) * x₁) + 38313
9           1.129e+10  -0.000e+00  y = (x₁ * x₀) * ((x₀ * -0.00090611) + 1.5175)
10          9.190e+07  -0.000e+00  y = ((x₀ * inv(x₁ + -982.95)) + x₁) * x₀
11          7.972e+06  5.332e-01  y = x₀ * (inv(-0.054014 + sin(x₀)) + x₁)
13          4.205e+06  -0.000e+00  y = x₁ * (x₀ + inv((sin(x₀) * -9.606) + 0.50798))
15          3.842e+06  -0.000e+00  y = (x₀ + (inv((sin(x₀) * -9.606) + 0.50798) + -0.96585))...
                       

[ Info: Final population:
[ Info: Results saved to:


{0: PySRRegressor.equations_ = [
 	    pick     score                                           equation  \
 	0         0.000000                                          229870.23   
 	1         0.496819                                            x1 * x0   
 	2         0.079785                              (x1 * 0.8534426) * x0   
 	3         0.024371               ((x0 * 0.77066994) * x1) + 38313.215   
 	4         0.093507    ((x0 * -0.00090613356) + 1.5175121) * (x1 * x0)   
 	5         4.810631             ((x0 * inv(x1 + -982.9535)) + x1) * x0   
 	6         2.446648            x0 * (inv(sin(x0) + -0.054012753) + x1)   
 	7   >>>>  0.357153  x1 * (inv(1.6555322 + (-31.26067 * sin(x0))) +...   
 	8         0.030416  (x0 + (-0.82677305 + inv((sin(x0) * -20.656754...   
 	9         0.001398  (x1 + 0.24277031) * ((inv((sin(x0) * -20.65675...   
 	10        0.133837  (x0 + (inv((x1 * (sin(x1) * inv(x0))) + -1.018...   
 	11        0.056327  x1 * (x0 + ((inv((sin(x1) * (x1 * inv(x0))) +

  - SR_output/block_17429733968/dim0_1764339129/hall_of_fame.csv


In [22]:
symbolic_model_multiplication.show_symbolic_expression()


➡️ Standard symbolic expressions for output dimension 0:
    complexity          loss  \
0            1  4.526662e+10   
1            3  1.675895e+10   
2            5  1.428719e+10   
3            7  1.360751e+10   
4            9  1.128650e+10   
5           10  9.190282e+07   
6           11  7.957250e+06   
7           13  3.895329e+06   
8           15  3.665433e+06   
9           17  3.655202e+06   
10          18  3.197324e+06   
11          20  2.856680e+06   
12          22  2.851968e+06   
13          25  2.789822e+06   
14          27  2.780307e+06   
15          29  2.776162e+06   
16          30  2.729567e+06   

                                             equation     score  \
0                                           229870.23  0.000000   
1                                             x1 * x0  0.496819   
2                               (x1 * 0.8534426) * x0  0.079785   
3                ((x0 * 0.77066994) * x1) + 38313.215  0.024371   
4     ((x0 * -0.00090613356) +

## Counting

What does the LLM return when counting the number of 1s in a string of 1s and 0s?

In [27]:
extract_boxed_number(llm_call("Return only the numeric answer in the format $boxed$. How many 1s are there in the string 000101", max_tokens= 250))

4.0

In [28]:
def random_number_string_01(N = 100, len_sequence = 4):
    return np.random.randint(0, 2, size=(N, len_sequence))

X_counts_01 = random_number_string_01(N = 25)

In [29]:
def llm_counting(X):
    outputs = []
    # X is of shape (N,10)
    for n in range(X.shape[0]):
        sequence = ''.join(map(str, X[n,:]))
        # print(sequence)
        output = llm_call(f"Return only the numeric answer in the format $boxed$. How many 1s are there in the string {sequence}", max_tokens=250)
        # print(f"Return only the numeric answer in the format $boxed$. How many 1s are there in the string {sequence}")

        try:
            output = extract_boxed_number(output)
        except ValueError:
            print("No boxed number found. Trying again with more tokens.")
            output = llm_call(f"Return only the numeric answer in the format $boxed$. How many 1s are there in the string {sequence}", max_tokens=500)
            output = extract_boxed_number(output)
        outputs.append(output)
    return np.array(outputs)

In [30]:
# Initialise our model
symbolic_model_counting = SymbolicModel(llm_counting)

No name specified for this block. Label is block_14826965584.


In [31]:
symbolic_model_counting.distill(X_counts_01, sr_params=sr_params)

🛠️ Running SR on output dimension 0 of 0


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 1.970e+06
Progress: 11800 / 31000 total iterations (38.065%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           1.434e+00  0.000e+00  y = 2.92
3           1.206e+00  8.627e-02  y = x₂ + 2.44
5           1.078e+00  5.608e-02  y = (x₂ + 2.04) + x₁
7           1.038e+00  1.924e-02  y = (x₀ * (x₁ + -1.0961)) + 3.25
8           7.888e-01  2.743e-01  y = (x₂ * inv(x₃ + 0.43874)) + 2.2695
9           7.258e-01  8.321e-02  y = (((x₃ * -2.0286) + 2.1385) * x₂) + 2.4615
10          7.044e-01  2.995e-02  y = (inv(x₃ + 0.43035) * x₂) + (x₁ + 1.8595)
11          6.458e-01  8.684e-02  y = (x₂ * ((x₃ * (x₁ + -2.1716)) + 2.1384)) + 2.4616
12          4.649e-01  3.287e-01  y = (x₂ * inv(x₀ + (x₃ + 0.32121))) + (x₁ + 1.9131)
13          3.969e-01  1.581e-01  y = ((((x

[ Info: Final population:
[ Info: Results saved to:


{0: PySRRegressor.equations_ = [
 	    pick     score                                           equation  \
 	0         0.000000                                          2.9200118   
 	1         0.086274                                      x2 + 2.439992   
 	2         0.056081                              (x2 + 2.0399914) + x1   
 	3         0.019240                (x0 * (x1 + -1.096135)) + 3.2499862   
 	4         0.274270            (x2 * inv(x3 + 0.43874383)) + 2.2695189   
 	5         0.083207   (((x3 * -2.028587) + 2.138481) * x2) + 2.4615328   
 	6         0.029947      (inv(x3 + 0.43034637) * x2) + (x1 + 1.859502)   
 	7         0.086838  (x2 * ((x3 * (x1 + -2.171553)) + 2.1383862)) +...   
 	8         0.328665  (x2 * inv(x0 + (x3 + 0.32121232))) + (x1 + 1.9...   
 	9         0.158085  ((((x0 + -1.7461581) + x3) * (x2 * -1.6666652)...   
 	10        0.271281  ((((x2 * -2.648921) + 0.9822007) * (x3 + (x0 +...   
 	11        0.371268  ((x1 + -3.2856712) * (((x2 + -0.4517958) * ((

  - SR_output/block_14826965584/dim0_1764340041/hall_of_fame.csv


The LLM is really terrible at counting! The equations it learns are not remotely what you would expect ($x_0+x_1+...+x_N$).

In [32]:
symbolic_model_counting.show_symbolic_expression()


➡️ Standard symbolic expressions for output dimension 0:
    complexity      loss                                           equation  \
0            1  1.433600                                          2.9200118   
1            3  1.206400                                      x2 + 2.439992   
2            5  1.078400                              (x2 + 2.0399914) + x1   
3            7  1.037692                (x0 * (x1 + -1.096135)) + 3.2499862   
4            8  0.788777            (x2 * inv(x3 + 0.43874383)) + 2.2695189   
5            9  0.725802   (((x3 * -2.028587) + 2.138481) * x2) + 2.4615328   
6           10  0.704389      (inv(x3 + 0.43034637) * x2) + (x1 + 1.859502)   
7           11  0.645802  (x2 * ((x3 * (x1 + -2.171553)) + 2.1383862)) +...   
8           12  0.464902  (x2 * inv(x0 + (x3 + 0.32121232))) + (x1 + 1.9...   
9           13  0.396923  ((((x0 + -1.7461581) + x3) * (x2 * -1.6666652)...   
10          15  0.230714  ((((x2 * -2.648921) + 0.9822007) * (x3 + (x0 +.

## Temperature conversion

Let's see how the LLM calculates Celsius to Fahrenheit. We would expect $y = \frac{9}{5}x + 32$.

In [33]:
llm_call("Return only the numeric answer in the format $boxed$. What is 30 degrees Celsius in Fahrenheit?")

"To convert Celsius to Fahrenheit, multiply the Celsius temperature by 9/5 and add 32. Here's the formula: $F = \\frac{9}{5}C + 32$ where $C$ is the temperature in Celsius. Plug in the value of $C$ and solve for $F$. $F = \\frac{9}{5}(30) + 32$ $F = \\frac{270}{5} + 32$ $F = 54 + 32$ $F = 86$ Therefore, 30 degrees Celsius is equal to 86 degrees Fahrenheit."

First, let's try with temperatures that are within a regular range (ie. between -20 and 200C).

In [34]:
def llm_C_to_F(X):
    outputs = []
    # X is of shape (N,1)
    for n in range(X.shape[0]):
        temp_C = X[n,0]
        output = llm_call(f"Return only the numeric answer in the format $boxed$. What is {int(temp_C)} degreees Celsius in Fahrenheit?")
        try:
            output = extract_boxed_number(output)
        except ValueError:
            print("No boxed number found. Trying again with more tokens.")
            output = llm_call(f"Return only the numeric answer in the format $boxed$. What is {int(temp_C)} degreees Celsius in Fahrenheit?", max_tokens= 500)
            output = extract_boxed_number(output)
        outputs.append(output)
    return np.array(outputs)

In [35]:
def random_numbers(N = 100, minimum = 0, maximum = 999):
    return np.random.randint(minimum, maximum, size=(N, 1))

In [36]:
X_temps = random_numbers (N = 50, minimum=-20, maximum=200)

In [37]:
# Initialise our model
symbolic_model_C_to_F = SymbolicModel(llm_C_to_F)

No name specified for this block. Label is block_17429614352.


In [38]:
symbolic_model_C_to_F.distill(X_temps, sr_params=sr_params)

🛠️ Running SR on output dimension 0 of 0


/Users/liz/PhD/SymTorch_project/symtorch_venv/lib/python3.11/site-packages/pysr/sr.py:2811: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 2.010e+06
Progress: 10198 / 31000 total iterations (32.897%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           5.150e+04  0.000e+00  y = 213.54
3           3.533e+04  1.856e-01  y = x₀ * 2.2271
5           3.522e+04  -0.000e+00  y = (x₀ * 2.0842) + 19.069
7           3.515e+04  -0.000e+00  y = (x₀ * (x₀ * 0.010869)) + 78.179
8           6.494e+03  1.686e+00  y = x₀ * (inv(x₀ + -168.86) + 1.9003)
10          6.086e+03  3.212e-02  y = (x₀ + 16.528) * (inv(x₀ + -168.85) + 1.7258)
12          5.954e+03  1.076e-02  y = x₀ + ((x₀ * (inv(x₀ + -168.86) + 0.57332)) + 42.782)
14          5.948e+03  2.451e-04  y = (x₀ * (inv(x₀ + -168.86) + -0.394)) + ((x₀ + x₀) + 39....
                                      498)
16          5.948e+03  -0.000e+00  y = ((inv(x₀

[ Info: Final population:
[ Info: Results saved to:


{0: PySRRegressor.equations_ = [
 	    pick     score                                           equation  \
 	0         0.000000                                           213.5406   
 	1         0.188442                                     x0 * 2.2270966   
 	2         0.001546                       (x0 * 2.0842118) + 19.069176   
 	3         0.001028                 ((x0 * 0.01086909) * x0) + 78.1772   
 	4         1.688924            x0 * (inv(x0 + -168.86307) + 1.8887489)   
 	5         0.042564  ((inv(x0 + -168.86176) + 1.6329502) * x0) + 37...   
 	6         0.000728  x0 + ((x0 * (inv(x0 + -168.86124) + 0.5733208)...   
 	7         0.596715  x0 * (inv(x0 + -168.86176) + (inv(x0 + -51.868...   
 	8         0.037221  (((inv(x0 + -168.86482) + inv(x0 + -51.86347))...   
 	9         0.000192  (((inv(x0 + -51.864178) + 1.6464747) + inv(x0 ...   
 	10        0.486669  x0 * (((inv(-129.83676 + x0) + inv(x0 + -51.86...   
 	11  >>>>  0.138889  ((((inv(x0 + -168.86176) + 1.6776766) + inv(x

  - SR_output/block_17429614352/dim0_1764340979/hall_of_fame.csv


In [39]:
symbolic_model_C_to_F.show_symbolic_expression(complexity=5)


➡️ Dimension 0 - Complexity 5:
   (x0 * 2.0842118) + 19.069176 (loss: 3.521945e+04)
